# 02 — Feature Engineering

Build ML-ready features from the cleaned Cameroon agricultural dataset.

**Feature groups:**
1. Temporal features (season, month, day-of-year encoding)
2. Climate indices (GDD, aridity, VPD proxy)
3. Soil fertility composite index
4. Input intensity score
5. Agroecological zone encoding
6. Interaction features
7. Crop group encoding
8. Export feature-engineered dataset

In [ ]:
import pandas as pd
import numpy as np
import sys, os
from datasets import load_dataset

sys.path.insert(0, os.path.abspath('..'))
from utils.constants import AGROECOLOGICAL_ZONES

CLEAN_PATH = '../data/generated/cameroon_agricultural_cleaned.csv'
FE_PATH = '../data/generated/cameroon_agricultural_features.csv'

# Load cleaned dataset: prefer local file (output of 01), fallback to Hugging Face
if os.path.exists(CLEAN_PATH):
    df = pd.read_csv(CLEAN_PATH, parse_dates=['observation_date'])
    print(f'Loaded from local: {CLEAN_PATH}')
else:
    HF_REPO = 'synthi-ai/cameroon-agricultural-data'
    print(f'Local file not found, loading from Hugging Face: {HF_REPO}')
    ds = load_dataset(HF_REPO, split='train')
    df = ds.to_pandas()
    RENAME_MAP = {
        'yield_grain': 'yield_kg_ha',
        'biomass_total': 'biomass_kg_ha',
        'season_type': 'season',
    }
    df = df.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df.columns})
    CROP_GROUPS = {
        'maize': 'cereals', 'rice': 'cereals', 'sorghum': 'cereals', 'millet': 'cereals', 'wheat': 'cereals',
        'cowpea': 'legumes', 'groundnut': 'legumes', 'common_bean': 'legumes', 'soybean': 'legumes',
        'cassava': 'root_tubers', 'sweet_potato': 'root_tubers', 'yam': 'root_tubers', 'potato': 'root_tubers',
        'tomato': 'vegetables', 'pepper': 'vegetables', 'onion': 'vegetables', 'cabbage': 'vegetables',
        'carrot': 'vegetables', 'lettuce': 'vegetables', 'cucumber': 'vegetables', 'okra': 'vegetables',
        'cocoa': 'tree_crops', 'coffee': 'tree_crops', 'oil_palm': 'tree_crops', 'mango': 'tree_crops',
        'plantain_banana': 'tree_crops', 'pineapple': 'tree_crops', 'papaya': 'tree_crops',
    }
    if 'crop_group' not in df.columns and 'crop_name' in df.columns:
        df['crop_group'] = df['crop_name'].map(CROP_GROUPS)
    if 'observation_date' in df.columns:
        df['observation_date'] = pd.to_datetime(df['observation_date'])

print(f'Loaded {len(df):,} rows x {df.shape[1]} cols')
df.head(3)

## 1 — Temporal features

In [ ]:
df['month'] = df['observation_date'].dt.month
df['day_of_year'] = df['observation_date'].dt.dayofyear
df['year'] = df['observation_date'].dt.year

# Cyclical encoding of month and day_of_year
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12).round(4)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12).round(4)
df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365).round(4)
df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365).round(4)

# Season numerical encoding (ordinal by rainfall intensity)
SEASON_ORDER = {
    'grand_dry_season': 0, 'dry_season': 0,
    'early_rainy': 1, 'transition_to_dry': 1,
    'first_rainy_season': 2, 'rainy_season': 2,
    'petit_dry_season': 3,
    'second_rainy_season': 4, 'late_rainy': 4,
    'off_season': 0,
}
df['season_ordinal'] = df['season'].map(SEASON_ORDER).fillna(0).astype('int8')

# Binary: is it a rainy season?
rainy = {'first_rainy_season','second_rainy_season','rainy_season','early_rainy','late_rainy'}
df['is_rainy_season'] = df['season'].isin(rainy).astype('int8')

print(f'Temporal features added: month, day_of_year, year, sin/cos encodings, season_ordinal, is_rainy_season')
df[['observation_date','month','month_sin','month_cos','season','season_ordinal','is_rainy_season']].head(5)

## 2 — Climate indices

In [ ]:
# Growing Degree Days (base 10)
df['gdd_base10'] = (df['temperature_mean'] - 10).clip(lower=0).round(1)

# Growing Degree Days (base 15 — tropical crops)
df['gdd_base15'] = (df['temperature_mean'] - 15).clip(lower=0).round(1)

# Diurnal temperature range
df['diurnal_range'] = (df['temperature_max'] - df['temperature_min']).round(1)

# Simple aridity proxy: precipitation / (temperature_mean + 10)
# Higher = wetter conditions
df['aridity_index'] = (df['precipitation_daily'] / (df['temperature_mean'] + 10).clip(lower=1)).round(3)

# VPD proxy (simplified): higher temp + lower humidity = higher VPD = more stress
# VPD ~ 0.6108 * exp(17.27*T/(T+237.3)) * (1 - RH/100)
sat_vp = 0.6108 * np.exp(17.27 * df['temperature_mean'] / (df['temperature_mean'] + 237.3))
df['vpd_kpa'] = (sat_vp * (1 - df['relative_humidity']/100)).round(3)

# Temperature-altitude residual (deviation from expected lapse rate)
# Expected temp at sea level: temp + 6.5 * elevation/1000
df['temp_altitude_residual'] = (df['temperature_mean'] + 6.5 * df['elevation']/1000).round(1)

# Precipitation intensity (binary: heavy rain day > 20mm)
df['heavy_rain_day'] = (df['precipitation_daily'] > 20).astype('int8')

print('Climate indices added: gdd_base10/15, diurnal_range, aridity_index, vpd_kpa, temp_altitude_residual, heavy_rain_day')
df[['temperature_mean','gdd_base10','diurnal_range','aridity_index','vpd_kpa']].describe().round(3)

## 3 — Soil fertility composite

In [ ]:
# Normalize each soil variable to [0, 1] range then average
soil_cols = ['ph_water', 'organic_carbon', 'total_nitrogen', 'available_phosphorus']

soil_norm = pd.DataFrame()
for c in soil_cols:
    cmin, cmax = df[c].min(), df[c].max()
    if cmax > cmin:
        soil_norm[c] = (df[c] - cmin) / (cmax - cmin)
    else:
        soil_norm[c] = 0.5

# pH is optimal around 6.0-6.5 for most crops; penalize extremes
ph_optimal = 6.25
soil_norm['ph_water'] = 1 - (abs(df['ph_water'] - ph_optimal) / 3.0).clip(0, 1)

df['soil_fertility_index'] = soil_norm.mean(axis=1).round(3)

# C:N ratio (lower = faster mineralization, generally better)
df['cn_ratio'] = (df['organic_carbon'] / df['total_nitrogen'].clip(lower=0.01)).round(1)

# Clay activity proxy: CEC estimate ~ clay% * 0.5 + OC * 2
df['cec_estimate'] = (df['clay_percentage'] * 0.5 + df['organic_carbon'] * 2).round(1)

print('Soil features added: soil_fertility_index, cn_ratio, cec_estimate')
df[['soil_fertility_index','cn_ratio','cec_estimate']].describe().round(2)

## 4 — Input intensity score

In [ ]:
# Normalize fertilizer inputs and combine into an intensity score
n_max = df['fertilizer_nitrogen'].max() or 1
p_max = df['fertilizer_phosphorus'].max() or 1
o_max = df['organic_fertilizer'].max() or 1

df['input_intensity'] = (
    0.4 * df['fertilizer_nitrogen'] / n_max +
    0.2 * df['fertilizer_phosphorus'] / p_max +
    0.2 * df['organic_fertilizer'] / o_max +
    0.2 * df['irrigation_applied'].astype(float)
).round(3)

# Total mineral fertilizer
df['total_mineral_fert'] = (df['fertilizer_nitrogen'] + df['fertilizer_phosphorus']).round(1)

# Organic-to-mineral ratio
df['organic_mineral_ratio'] = (
    df['organic_fertilizer'] / df['total_mineral_fert'].clip(lower=1)
).round(3)

print('Input features added: input_intensity, total_mineral_fert, organic_mineral_ratio')
df[['input_intensity','total_mineral_fert','organic_mineral_ratio']].describe().round(3)

## 5 — Zone & crop encoding

In [ ]:
# One-hot encode agroecological zone
zone_dummies = pd.get_dummies(df['agroecological_zone'], prefix='zone', dtype='int8')
df = pd.concat([df, zone_dummies], axis=1)

# One-hot encode crop group
group_dummies = pd.get_dummies(df['crop_group'], prefix='grp', dtype='int8')
df = pd.concat([df, group_dummies], axis=1)

# Latitude regime: bimodal (0) vs monomodal (1)
df['rainfall_regime'] = (df['latitude'] >= 6.0).astype('int8')

# Altitude class
df['altitude_class'] = pd.cut(
    df['elevation'],
    bins=[0, 200, 800, 1200, 2500, 5000],
    labels=['lowland', 'plateau', 'mid_altitude', 'highland', 'mountain']
).astype(str)
alt_dummies = pd.get_dummies(df['altitude_class'], prefix='alt', dtype='int8')
df = pd.concat([df, alt_dummies], axis=1)

print(f'Zone dummies: {list(zone_dummies.columns)}')
print(f'Group dummies: {list(group_dummies.columns)}')
print(f'Altitude dummies: {list(alt_dummies.columns)}')

## 6 — Interaction features

In [ ]:
# Humidity x temperature interaction (disease pressure proxy)
df['humidity_temp_interaction'] = (df['relative_humidity'] * df['temperature_mean'] / 100).round(2)

# Rainfall x soil organic carbon (nutrient leaching risk)
df['rain_oc_interaction'] = (df['precipitation_daily'] * df['organic_carbon']).round(2)

# Input intensity x soil fertility (expected response)
df['input_soil_interaction'] = (df['input_intensity'] * df['soil_fertility_index']).round(3)

# Irrigation x precipitation (supplemental water index)
df['water_supply_index'] = (
    df['precipitation_daily'] + df['irrigation_applied'].astype(float) * 30  # ~30mm equiv per irrigation event
).round(1)

# Disease risk score: humidity * temperature / (solar_radiation + 1)
df['disease_risk_score'] = (
    df['relative_humidity'] * df['temperature_mean'] / (df['solar_radiation'].clip(lower=1) * 100)
).round(3)

print('Interaction features added.')
df[['humidity_temp_interaction','rain_oc_interaction','input_soil_interaction','water_supply_index','disease_risk_score']].describe().round(3)

## 7 — Yield efficiency metrics

In [ ]:
# Nitrogen use efficiency (yield per kg N applied)
df['nue'] = (df['yield_kg_ha'] / df['fertilizer_nitrogen'].clip(lower=1)).round(1)

# Yield per unit water (rain + irrigation proxy)
df['wue'] = (df['yield_kg_ha'] / df['water_supply_index'].clip(lower=1)).round(1)

# Yield gap proxy: actual yield / zone mean yield
zone_mean_yield = df.groupby('agroecological_zone')['yield_kg_ha'].transform('mean')
df['yield_gap_ratio'] = (df['yield_kg_ha'] / zone_mean_yield.clip(lower=1)).round(3)

print('Efficiency features added: nue, wue, yield_gap_ratio')
df[['nue','wue','yield_gap_ratio']].describe().round(1)

## 8 — Feature summary & export

In [ ]:
new_features = [
    'month','day_of_year','year','month_sin','month_cos','doy_sin','doy_cos',
    'season_ordinal','is_rainy_season',
    'gdd_base10','gdd_base15','diurnal_range','aridity_index','vpd_kpa',
    'temp_altitude_residual','heavy_rain_day',
    'soil_fertility_index','cn_ratio','cec_estimate',
    'input_intensity','total_mineral_fert','organic_mineral_ratio',
    'rainfall_regime',
    'humidity_temp_interaction','rain_oc_interaction','input_soil_interaction',
    'water_supply_index','disease_risk_score',
    'nue','wue','yield_gap_ratio',
]

print(f'Total columns: {df.shape[1]}')
print(f'New engineered features: {len(new_features)}')
print(f'One-hot columns: {len(zone_dummies.columns) + len(group_dummies.columns) + len(alt_dummies.columns)}')
print()
print('Feature list:')
for f in new_features:
    print(f'  {f}: {df[f].dtype}')

In [ ]:
# Drop intermediate text columns not needed for ML
drop_cols = ['record_id', 'location_id', 'altitude_class']
df_ml = df.drop(columns=[c for c in drop_cols if c in df.columns])

df_ml.to_csv(FE_PATH, index=False)
size_mb = os.path.getsize(FE_PATH) / (1024*1024)
print(f'Saved feature-engineered dataset: {len(df_ml):,} rows x {df_ml.shape[1]} cols, {size_mb:.1f} MB')
print(f'Path: {FE_PATH}')

In [ ]:
# Final correlation matrix for engineered features vs yield
target = 'yield_kg_ha'
corr_with_yield = df_ml[new_features + [target]].corr()[target].drop(target).sort_values(ascending=False)
print('Top 10 features correlated with yield_kg_ha:')
print(corr_with_yield.head(10).round(3))
print()
print('Bottom 5:')
print(corr_with_yield.tail(5).round(3))